# RA2CE -- Tahoe Basin Network Resilience Analysis

Runs three analyses using the [RA2CE](https://github.com/Deltares/ra2ce) (Risk Assessment to Climate Effects) framework against the Overture Drive Routable network for the Lake Tahoe Basin.

| Analysis | RA2CE Module | Output |
|---|---|---|
| **Single Link Redundancy (SLR)** | `SINGLE_LINK_REDUNDANCY` | Per-segment detour availability and extra distance |
| **OD Accessibility** | `OPTIMAL_ROUTE_ORIGIN_DESTINATION` | Baseline origin-to-destination routes |
| **OD + Hazard Disruption** | `MULTI_LINK_ORIGIN_CLOSEST_DESTINATION` | Accessibility loss under hazard exposure |

**Input network:** `Overture_Drive_Routable` from `Streets_Network.gdb`

**Run order:** Sections 1 and 2 (SLR) are standalone. Sections 3 and 4 require Origins/Destinations shapefiles and a hazard raster -- see notes in each section before running.

In [ ]:
# ============================================================
# Configuration -- edit these paths before running
# ============================================================
import os
from pathlib import Path

# PROJ data directory -- match to your conda environment.
# Not quite sure why this is needed but it won't work unless you do this. 
PROJ_DIR = r"C:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Library\share\proj"

# Input: Overture Drive Routable network (File GDB feature class)
NETWORK_GDB   = r"F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb"
NETWORK_LAYER = "Overture_Drive_Routable"

# OD analysis inputs -- built automatically in Section 3 prep cells
ORIGINS_GDB_LAYER      = "Origins"          # feature class in Streets_Network.gdb
DESTINATIONS_GDB_LAYER = "AutoEntryPoint"   # feature class in Streets_Network.gdb
ORIGINS_SHP            = "Origins.shp"      # exported shapefile written to static/network/
DESTINATIONS_SHP       = "Destinations.shp" # exported shapefile written to static/network/
ORIGINS_COUNT_FIELD    = "POPULATION"       # count/weight field; added synthetically if absent
EQUITY_WEIGHT_FIELD    = ""                 # per-origin equity weight column in Origins layer
                                            # (e.g. "SVI_SCORE", "EJ_WEIGHT"); set before Section 3 equity cell

# Hazard raster (place in static/network/ before running Section 4)
HAZARD_RASTER = "hazard.tif"   # filename within static/network/
HAZARD_CRS    = "EPSG:3857"    # native CRS of the raster

# ============================================================
# Derived paths -- no edits needed below
# ============================================================
ROOT_DIR     = Path(".")
NETWORK_PATH = ROOT_DIR / "static" / "network"
OUTPUT_PATH  = ROOT_DIR / "output"
NETWORK_SHP  = NETWORK_PATH / "overture_drive_routable.shp"

# PROJ environment
os.environ["PROJ_DATA"] = PROJ_DIR
os.environ["PROJ_LIB"]  = PROJ_DIR

import pyproj
pyproj.datadir.set_data_dir(PROJ_DIR)

# Logging
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# Core imports
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from pyproj import CRS

import ra2ce
from ra2ce.ra2ce_handler import Ra2ceHandler
from ra2ce.network.network_config_data.network_config_data import (
    NetworkSection, NetworkConfigData, OriginsDestinationsSection, HazardSection,
)
from ra2ce.network.network_config_data.enums.source_enum import SourceEnum
from ra2ce.network.network_config_data.enums.aggregate_wl_enum import AggregateWlEnum
from ra2ce.analysis.analysis_config_data.analysis_config_data import (
    AnalysisSectionLosses, AnalysisConfigData,
)
from ra2ce.analysis.analysis_config_data.enums.analysis_losses_enum import AnalysisLossesEnum
from ra2ce.analysis.analysis_config_data.enums.weighing_enum import WeighingEnum

print(f"ra2ce {ra2ce.__version__}  |  cwd: {Path('.').resolve()}")


In [ ]:
# ============================================================
# RA2CE bug patch: closest_node coordinate rounding
# ============================================================
# RA2CE rounds node coordinates to 7 decimal places when snapping OD points,
# breaking dict lookups in inverse_vertices_dict / inverse_nodes_dict when the
# full-precision keys no longer match (e.g. -120.28550210013262 -> -120.2855021).
# Replacing with a version that preserves full float64 precision fixes the issue.

import ra2ce.network.origins_destinations as _od_mod

def _closest_node_fixed(node: np.ndarray, nodes: np.ndarray) -> np.ndarray:
    deltas = nodes - node
    dist_2 = np.einsum("ij,ij->i", deltas, deltas)
    return nodes[np.where(dist_2 == dist_2.min())[0]][0]

_od_mod.closest_node = _closest_node_fixed
print("Applied closest_node precision patch.")

## 1. Prepare Network Input

Reads `Overture_Drive_Routable` from the File GDB and exports it to a shapefile that RA2CE can ingest. Skips the export on subsequent runs -- delete `static/network/overture_drive_routable.*` to force a refresh.

> **Note:** If the GDB is open in ArcGIS Pro, geopandas can still read it in read-only mode via fiona. Field names longer than 10 characters are truncated in the shapefile output (shapefile format limit), but RA2CE only requires the geometry for the analyses below.

In [ ]:
NETWORK_PATH.mkdir(parents=True, exist_ok=True)

if not NETWORK_SHP.exists():
    print(f"Reading '{NETWORK_LAYER}' from GDB ...")
    gdf = gpd.read_file(NETWORK_GDB, layer=NETWORK_LAYER)
    print(f"  {len(gdf):,} features  |  CRS: {gdf.crs}")
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)
        print("  Reprojected to EPSG:4326")
    gdf.to_file(NETWORK_SHP)
    print(f"  Exported -> {NETWORK_SHP.name}")
else:
    gdf = gpd.read_file(NETWORK_SHP)
    print(f"Using cached shapefile: {NETWORK_SHP.name}")
    print(f"  {len(gdf):,} features  |  CRS: {gdf.crs}")

## 2. Single Link Redundancy Analysis

For each road segment, RA2CE removes it from the graph and checks whether an alternative route still connects its endpoints. Key output fields:

- `detour` -- 1 if an alternative exists, 0 if the link is critical (no detour)
- `alt_length` -- length of the alternative route (m)
- `diff_length` -- extra travel distance vs. the direct route (m)

`reuse_network_output=True` skips graph rebuilding on subsequent runs.

In [ ]:
network_section = NetworkSection(
    source=SourceEnum.SHAPEFILE,
    primary_file=NETWORK_SHP,
    save_gpkg=True,
    reuse_network_output=True,
)

network_config = NetworkConfigData(
    root_path=ROOT_DIR,
    static_path=ROOT_DIR / "static",
    crs=CRS.from_epsg(4326),
    network=network_section,
)

slr_analysis = AnalysisSectionLosses(
    name="tahoe_slr",
    analysis=AnalysisLossesEnum.SINGLE_LINK_REDUNDANCY,
    weighing=WeighingEnum.LENGTH,
    save_csv=True,
    save_gpkg=True,
)

slr_config = AnalysisConfigData(
    root_path=ROOT_DIR,
    output_path=OUTPUT_PATH,
    static_path=ROOT_DIR / "static",
    analyses=[slr_analysis],
)

handler = Ra2ceHandler.from_config(network=network_config, analysis=slr_config)
handler.configure()
handler.run_analysis()

In [ ]:
slr_gpkg = OUTPUT_PATH / "single_link_redundancy" / "tahoe_slr.gpkg"
slr_gdf  = gpd.read_file(slr_gpkg)

n_total  = len(slr_gdf)
n_conn   = (slr_gdf["detour"] == 1).sum()
n_crit   = n_total - n_conn
print(f"Total edges  : {n_total:,}")
print(f"With detour  : {n_conn:,}  ({100*n_conn/n_total:.1f}%)")
print(f"Critical     : {n_crit:,}  ({100*n_crit/n_total:.1f}%)")
slr_gdf[["detour", "alt_length", "diff_length"]].describe()

In [ ]:
# ============================================================
# Export SLR results for ArcGIS Pro
# ============================================================
# ArcGIS Pro fails on RA2CE GeoPackages with:
#   "Invalid SQL syntax [near ',': syntax error]"
# Root cause: GDAL writes float column types as NUMERIC(15,7) -- the
# comma inside the precision specifier breaks ArcGIS Pro's SQL tokenizer.
# Fix: export as GeoJSON or Shapefile, which have no SQL layer at all.

slr_clean = slr_gdf.copy()

# Convert list-type columns to pipe-separated strings
def _to_str(v):
    if v is None:
        return ""
    try:
        if isinstance(v, float) and np.isnan(v):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, (list, tuple)):
        return "|".join(str(x) for x in v)
    if isinstance(v, bytes):
        return v.decode("utf-8", errors="replace")
    return str(v)

for col in ["alt_nodes", "match_ids"]:
    if col in slr_clean.columns:
        slr_clean[col] = slr_clean[col].apply(_to_str)
        print(f"  Converted '{col}' to string")

# Force all non-geometry columns to scalar types (avoids NUMERIC(p,s) in GPKG)
for col in slr_clean.columns:
    if col == "geometry":
        continue
    s = slr_clean[col]
    if s.dtype.kind == "f":
        slr_clean[col] = s.astype("float64")
    elif s.dtype.kind in ("i", "u"):
        slr_clean[col] = s.astype("int64")
    else:
        slr_clean[col] = s.fillna("").astype(str)

# Option A: GeoJSON -- no SQL layer, add via Catalog pane in ArcGIS Pro
out_json = OUTPUT_PATH / "single_link_redundancy" / "tahoe_slr.geojson"
slr_clean.to_file(out_json, driver="GeoJSON")
print(f"\nGeoJSON: {out_json.resolve()}")

# Option B: Shapefile -- universally compatible; diff_length -> diff_leng (10-char limit)
out_shp = OUTPUT_PATH / "single_link_redundancy" / "tahoe_slr.shp"
slr_clean.to_file(out_shp)
print(f"Shapefile: {out_shp.resolve()}")
print("  (diff_length truncated to diff_leng in .dbf)")

### SLR Maps

Three static visualizations saved to `output/`.

In [ ]:
# Map 1: Connectivity -- does a detour exist?
fig, ax = plt.subplots(figsize=(20, 20), facecolor="#1a1a2e")
ax.set_facecolor("#1a1a2e")

connected    = slr_gdf[slr_gdf["detour"] == 1]
disconnected = slr_gdf[slr_gdf["detour"] == 0]

connected.plot(   ax=ax, color="#81b29a", linewidth=0.7, alpha=0.9, label="Alternative route exists")
disconnected.plot(ax=ax, color="#e07a5f", linewidth=1.2, alpha=0.95, label="No alternative route")

ax.set_axis_off()
ax.set_title(
    "Tahoe Basin -- Single Link Redundancy\nAlternative Route Availability",
    color="#e8d5b7", fontsize=16, fontweight="bold", pad=15,
)
ax.legend(
    facecolor="#2a2a4e", edgecolor="none",
    labelcolor="#e8d5b7", fontsize=11, loc="lower right",
)

pct_conn = 100 * len(connected) / max(n_total, 1)
ax.text(
    0.02, 0.02,
    f"Connected: {len(connected):,} ({pct_conn:.1f}%)\nNo detour: {len(disconnected):,} ({100-pct_conn:.1f}%)",
    transform=ax.transAxes, color="#e8d5b7", fontsize=10,
    verticalalignment="bottom",
    bbox=dict(facecolor="#2a2a4e", edgecolor="none", alpha=0.7, pad=5),
)

plt.tight_layout()
out_path = OUTPUT_PATH / "tahoe_slr_connectivity.png"
fig.savefig(out_path, dpi=300, facecolor=fig.get_facecolor(), bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

In [ ]:
# Map 2: Detour extra distance (diff_length), clipped to 95th percentile
detour_gdf = slr_gdf[slr_gdf["detour"] == 1].copy()
no_detour  = slr_gdf[slr_gdf["detour"] == 0].copy()

diff_vals = detour_gdf["diff_length"].dropna()
vmin, vmax = 0, np.percentile(diff_vals, 95)

cmap = cm.plasma
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

fig, ax = plt.subplots(figsize=(20, 20), facecolor="#1a1a2e")
ax.set_facecolor("#1a1a2e")

no_detour.plot(ax=ax, color="#3a3a5e", linewidth=0.6, alpha=0.6)
detour_gdf.plot(
    ax=ax, column="diff_length",
    cmap=cmap, norm=norm,
    linewidth=0.8, alpha=0.9,
    missing_kwds={"color": "#3a3a5e"},
)

ax.set_axis_off()
ax.set_title(
    "Tahoe Basin -- Single Link Redundancy\nAdditional Detour Distance (m)",
    color="#e8d5b7", fontsize=16, fontweight="bold", pad=15,
)

sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02, orientation="vertical")
cbar.set_label("Extra detour distance (m)", color="#e8d5b7", fontsize=11)
cbar.ax.yaxis.set_tick_params(color="#e8d5b7")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="#d8a24c")
cbar.outline.set_edgecolor("#e8d5b7")

ax.text(
    0.02, 0.02,
    f"Median detour: {diff_vals.median():.0f} m\nMax detour (95th pct): {vmax:.0f} m",
    transform=ax.transAxes, color="#e8d5b7", fontsize=10,
    verticalalignment="bottom",
    bbox=dict(facecolor="#2a2a4e", edgecolor="none", alpha=0.7, pad=5),
)

plt.tight_layout()
out_path = OUTPUT_PATH / "tahoe_slr_detour_length.png"
fig.savefig(out_path, dpi=300, facecolor=fig.get_facecolor(), bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

In [ ]:
# Map 3: Combined -- detour distance (blue to amber) + critical links (red)
detour_gdf = slr_gdf[slr_gdf["detour"] == 1].copy()
no_detour  = slr_gdf[slr_gdf["detour"] == 0].copy()

diff_vals = detour_gdf["diff_length"].dropna()
vmin, vmax = 0, np.percentile(diff_vals, 95)

cmap = LinearSegmentedColormap.from_list("blue_amber", ["#1a6faf", "#e8a020"])
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

BG = "#ddd8cc"
fig, ax = plt.subplots(figsize=(20, 20), facecolor=BG)
ax.set_facecolor(BG)

no_detour.plot(ax=ax, color="#c0392b", linewidth=1.0, alpha=0.95, zorder=2)
detour_gdf.plot(
    ax=ax, column="diff_length",
    cmap=cmap, norm=norm,
    linewidth=0.7, alpha=0.9,
    missing_kwds={"color": "#aaaaaa"},
    zorder=3,
)

ax.set_axis_off()
ax.set_title(
    "Tahoe Basin -- Single Link Redundancy\nDetour Distance and Critical Links",
    color="#1a1a2e", fontsize=16, fontweight="bold", pad=15,
)

sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02, orientation="vertical")
cbar.set_label("Additional detour distance (m)", color="#1a1a2e", fontsize=11)
cbar.ax.yaxis.set_tick_params(color="#1a1a2e")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="#1a1a2e")
cbar.outline.set_edgecolor("#1a1a2e")

legend_elements = [
    Line2D([0], [0], color="#c0392b",           linewidth=2, label="No alternative route"),
    Line2D([0], [0], color=cmap(norm(vmin)),     linewidth=2, label="Shorter detour"),
    Line2D([0], [0], color=cmap(norm(vmax)),     linewidth=2, label="Longer detour"),
]
ax.legend(
    handles=legend_elements,
    facecolor="#f0ece4", edgecolor="#aaaaaa",
    labelcolor="#1a1a2e", fontsize=10, loc="lower right",
)

ax.text(
    0.02, 0.02,
    f"No detour: {len(no_detour):,} links\nMedian detour: {diff_vals.median():.0f} m",
    transform=ax.transAxes, color="#1a1a2e", fontsize=10,
    verticalalignment="bottom",
    bbox=dict(facecolor="#f0ece4", edgecolor="#aaaaaa", alpha=0.85, pad=5),
)

plt.tight_layout()
out_path = OUTPUT_PATH / "tahoe_slr_combined.png"
fig.savefig(out_path, dpi=300, facecolor=fig.get_facecolor(), bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

## 3. OD Accessibility Analysis (Baseline -- no hazard)

Finds the shortest path from each origin to its nearest destination.

**Origins** are read from the `Origins` feature class in `Streets_Network.gdb` (same GDB as the network).

**Destinations** are the seven major highway exit points from the Tahoe Basin, built programmatically in the prep cells below.

Run the two prep cells before the analysis cell. On subsequent runs the shapefiles are cached and the prep cells skip automatically. Update `ORIGINS_COUNT_FIELD` in the Configuration cell if your Origins layer uses a different weight column.

### Prepare OD Inputs

Run these two cells once before the analysis. Both shapefiles are cached in `static/network/` -- delete them to force a refresh.

In [ ]:
# ============================================================
# Prep 1 of 2: Origins -- read from GDB, reproject to EPSG:4326, export shapefile
# ============================================================
origins_shp = NETWORK_PATH / ORIGINS_SHP

if not origins_shp.exists():
    print(f"Reading '{ORIGINS_GDB_LAYER}' from GDB ...")
    orig_gdf = gpd.read_file(NETWORK_GDB, layer=ORIGINS_GDB_LAYER)
    print(f"  {len(orig_gdf):,} features  |  CRS: {orig_gdf.crs}")
    print(f"  Columns: {list(orig_gdf.columns)}")

    if orig_gdf.crs is None or orig_gdf.crs.to_epsg() != 4326:
        orig_gdf = orig_gdf.to_crs(4326)
        print("  Reprojected to EPSG:4326")

    # RA2CE requires the count/weight column to exist
    if ORIGINS_COUNT_FIELD not in orig_gdf.columns:
        print(f"  '{ORIGINS_COUNT_FIELD}' not found -- adding synthetic count of 1")
        print(f"  Available columns: {[c for c in orig_gdf.columns if c != 'geometry']}")
        print(f"  Update ORIGINS_COUNT_FIELD in config if a different field should be used.")
        orig_gdf[ORIGINS_COUNT_FIELD] = 1
    else:
        print(f"  Count field '{ORIGINS_COUNT_FIELD}': sum = {orig_gdf[ORIGINS_COUNT_FIELD].sum():,}")

    orig_gdf.to_file(origins_shp)
    print(f"  Exported -> {origins_shp.name}")
else:
    orig_gdf = gpd.read_file(origins_shp)
    print(f"Using cached Origins shapefile: {origins_shp.name}")
    print(f"  {len(orig_gdf):,} features  |  CRS: {orig_gdf.crs}")

In [ ]:
# ============================================================
# Prep 2 of 2: Destinations -- read AutoEntryPoint from GDB
# ============================================================
# Reads the AutoEntryPoint feature class (highway/basin exit points)
# from Streets_Network.gdb, reprojects to EPSG:4326, and exports to
# static/network/Destinations.shp.
# Delete Destinations.shp to force a rebuild from the GDB.

destinations_shp = NETWORK_PATH / DESTINATIONS_SHP

if not destinations_shp.exists():
    print(f"Reading '{DESTINATIONS_GDB_LAYER}' from GDB ...")
    dest_gdf = gpd.read_file(NETWORK_GDB, layer=DESTINATIONS_GDB_LAYER)
    print(f"  {len(dest_gdf):,} features  |  CRS: {dest_gdf.crs}")
    print(f"  Columns: {[c for c in dest_gdf.columns if c != 'geometry']}")

    if dest_gdf.crs is None or dest_gdf.crs.to_epsg() != 4326:
        dest_gdf = dest_gdf.to_crs(4326)
        print("  Reprojected to EPSG:4326")

    # Keep only Point geometries (RA2CE destination requirement)
    dest_gdf = dest_gdf[dest_gdf.geometry.geom_type == "Point"].copy().reset_index(drop=True)
    print(f"  Point features kept: {len(dest_gdf):,}")

    dest_gdf.to_file(destinations_shp)
    print(f"  Exported -> {destinations_shp.name}")
else:
    dest_gdf = gpd.read_file(destinations_shp)
    print(f"Using cached Destinations: {destinations_shp.name}")
    print(f"  {len(dest_gdf):,} features  |  CRS: {dest_gdf.crs}")

# Print location summary
name_col = next(
    (c for c in dest_gdf.columns if c.lower() in ("name", "label", "id", "objectid", "fid")),
    None,
)
for _, row in dest_gdf.iterrows():
    label = row[name_col] if name_col else "Exit"
    print(f"  {label}: {row.geometry.y:.4f}N, {abs(row.geometry.x):.4f}W")


In [ ]:
# Optional: reproject Origins / Destinations to EPSG:4326
# Uncomment and run if your OD shapefiles are in a projected CRS.

# for stem, src_name in [("Origins", ORIGINS_SHP), ("Destinations", DESTINATIONS_SHP)]:
#     src = NETWORK_PATH / src_name
#     dst = NETWORK_PATH / src_name.replace(".shp", "_4326.shp")
#     gdf_od = gpd.read_file(src)
#     print(f"{src_name} CRS: {gdf_od.crs}")
#     if gdf_od.crs is None or gdf_od.crs.to_epsg() != 4326:
#         gdf_od.to_crs(4326).to_file(dst)
#         print(f"  Saved reprojected -> {dst.name}")
#         print(f"  Update ORIGINS_SHP / DESTINATIONS_SHP in config to use the _4326 copies.")
#     else:
#         print(f"  Already EPSG:4326 -- no conversion needed")

In [ ]:
network_section_od = NetworkSection(
    source=SourceEnum.SHAPEFILE,
    primary_file=NETWORK_SHP,
    save_gpkg=True,
    reuse_network_output=True,
)

od_section = OriginsDestinationsSection(
    origins=NETWORK_PATH / ORIGINS_SHP,
    destinations=NETWORK_PATH / DESTINATIONS_SHP,
    origin_count=ORIGINS_COUNT_FIELD,
)

network_config_od = NetworkConfigData(
    root_path=ROOT_DIR,
    static_path=ROOT_DIR / "static",
    crs=CRS.from_epsg(4326),
    network=network_section_od,
    origins_destinations=od_section,
)

od_analysis = AnalysisSectionLosses(
    name="tahoe_od_baseline",
    analysis=AnalysisLossesEnum.OPTIMAL_ROUTE_ORIGIN_DESTINATION,
    weighing=WeighingEnum.LENGTH,
    calculate_route_without_disruption=True,
    save_csv=True,
    save_gpkg=True,
)

od_config = AnalysisConfigData(
    output_path=OUTPUT_PATH,
    static_path=ROOT_DIR / "static",
    analyses=[od_analysis],
)

handler_od = Ra2ceHandler.from_config(network=network_config_od, analysis=od_config)
handler_od.configure()
handler_od.run_analysis()

In [ ]:
od_gpkg = OUTPUT_PATH / "optimal_route_origin_destination" / "tahoe_od_baseline.gpkg"
od_gdf  = gpd.read_file(od_gpkg)
print(f"OD routes: {len(od_gdf):,}  |  Columns: {list(od_gdf.columns)}")
od_gdf.head()

In [ ]:
# ============================================================
# OD Baseline Map: route color = distance, line width = population served
# ============================================================
import pandas as _pd

# od_gdf from the route GPKG carries only route geometry columns -- POPULATION
# lives in origin_destination_table.feather keyed by o_id.  Join it in so
# the population-weighted line-width logic can find it.
_od_feather = ROOT_DIR / "static" / "output_graph" / "origin_destination_table.feather"
if _od_feather.exists():
    _pop_lookup = (
        _pd.read_feather(_od_feather)
        .drop_duplicates("o_id")
        .set_index("o_id")[ORIGINS_COUNT_FIELD]
    )
    od_gdf = od_gdf.copy()
    od_gdf[ORIGINS_COUNT_FIELD] = od_gdf["origin"].map(_pop_lookup)
    print(f"Joined {ORIGINS_COUNT_FIELD} to OD routes  "
          f"(non-null: {od_gdf[ORIGINS_COUNT_FIELD].notna().sum():,}/{len(od_gdf):,})")
else:
    print(f"Warning: feather not found at {_od_feather} -- line widths will be uniform")

# Route length column
len_col = next(
    (c for c in od_gdf.columns
     if any(k in c.lower() for k in ("opt_dist", "distance", "length", "dist"))
     and "alt" not in c.lower()
     and c in od_gdf.select_dtypes("number").columns),
    None,
)
pop_col = ORIGINS_COUNT_FIELD if ORIGINS_COUNT_FIELD in od_gdf.columns else None

print(f"Length column: '{len_col}'  |  Population column: '{pop_col}'")
if len_col is None:
    raise ValueError("Could not detect a route-length column.")

# Keep only route geometries
routes = od_gdf[
    od_gdf.geometry.notna() &
    od_gdf.geometry.geom_type.isin(["LineString", "MultiLineString"])
].copy()

routes["_dist_km"] = routes[len_col].fillna(0) / 1000
vmax = float(np.percentile(routes["_dist_km"][routes["_dist_km"] > 0], 95))
norm_c = mcolors.Normalize(vmin=0, vmax=vmax)
cmap_r = cm.plasma

# Bin population into 5 quintile tiers for line width
if pop_col and pop_col in routes.columns and routes[pop_col].notna().any():
    pop_vals  = routes[pop_col].fillna(1).clip(lower=1)
    bins      = np.percentile(pop_vals, [0, 20, 40, 60, 80])
    lw_map    = {0: 0.4, 1: 0.9, 2: 1.7, 3: 2.7, 4: 4.0}
    routes["_lw"] = [lw_map[int(np.digitize(v, bins[1:]))] for v in pop_vals]
    person_km = (routes["_dist_km"] * pop_vals).sum()
    pop_note  = f"Total person-km: {person_km:,.0f}"
else:
    routes["_lw"] = 1.2
    lw_map    = None
    pop_note  = "(population column not found -- uniform line width)"
    print(pop_note)

BG = "#1a1a2e"
fig, ax = plt.subplots(figsize=(20, 20), facecolor=BG)
ax.set_facecolor(BG)

for lw_val in sorted(routes["_lw"].unique()):
    routes[routes["_lw"] == lw_val].plot(
        ax=ax, column="_dist_km",
        cmap=cmap_r, norm=norm_c,
        linewidth=lw_val, alpha=0.8,
        missing_kwds={"color": "#333"},
    )

# Destination exit markers -- pick the best available label column
try:
    _label_col = next(
        (c for c in dest_gdf.columns
         if c.lower() in ("name", "label", "exit_name", "highway", "route", "description")),
        next((c for c in dest_gdf.columns if c != "geometry"), None),
    )
    dest_gdf.plot(ax=ax, color="#f7c948", marker="*", markersize=200, zorder=5)
    for _, row in dest_gdf.iterrows():
        label = str(row[_label_col]) if _label_col else "Exit"
        ax.annotate(
            label,
            xy=(row.geometry.x, row.geometry.y),
            xytext=(5, 3), textcoords="offset points",
            color="#f7c948", fontsize=7.5, fontweight="bold",
        )
except NameError:
    pass

ax.set_axis_off()
ax.set_title(
    "Tahoe Basin -- OD Accessibility Baseline\n"
    "Line color = route length  |  Line width = population served",
    color="#e8d5b7", fontsize=15, fontweight="bold", pad=15,
)

sm = cm.ScalarMappable(cmap=cmap_r, norm=norm_c)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Route length (km)", color="#e8d5b7", fontsize=11)
cbar.ax.yaxis.set_tick_params(color="#e8d5b7")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="#e8d5b7")
cbar.outline.set_edgecolor("#e8d5b7")

if lw_map:
    lw_legend = [
        Line2D([0], [0], color="#aaaaaa", linewidth=lw_map[q], label=f"Q{q+1} population")
        for q in sorted(lw_map)
    ]
    ax.legend(
        handles=lw_legend, title="Population quintile",
        facecolor="#2a2a4e", edgecolor="none",
        labelcolor="#e8d5b7", title_fontsize=10, fontsize=9,
        loc="lower right",
    )

ax.text(
    0.02, 0.02,
    f"Routes: {len(routes):,}\n"
    f"Median length: {routes['_dist_km'].median():.1f} km\n"
    f"{pop_note}",
    transform=ax.transAxes, color="#e8d5b7", fontsize=10,
    verticalalignment="bottom",
    bbox=dict(facecolor="#2a2a4e", edgecolor="none", alpha=0.7, pad=5),
)

plt.tight_layout()
out_path = OUTPUT_PATH / "tahoe_od_routes.png"
fig.savefig(out_path, dpi=300, facecolor=fig.get_facecolor(), bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

# Export for ArcGIS Pro as Shapefile
def _clean_for_shp(gdf):
    out = gdf.copy()
    for col in out.columns:
        if col == "geometry":
            continue
        if out[col].dtype == object:
            def _to_str(v):
                if v is None: return ""
                if isinstance(v, (list, tuple)): return "|".join(str(x) for x in v)
                return str(v)
            out[col] = out[col].apply(_to_str)
        elif out[col].dtype.kind == "f":
            out[col] = out[col].astype("float64")
        elif out[col].dtype.kind in ("i", "u"):
            out[col] = out[col].astype("int64")
    return out
out_shp = OUTPUT_PATH / "tahoe_od_routes.shp"
_clean_for_shp(routes).to_file(out_shp)
print(f"Shapefile: {out_shp.resolve()}")


In [ ]:
# ============================================================
# Utilitarian Ranking -- population-weighted link criticality
# ============================================================
# Implements the utilitarian principle from RA2CE equity analysis:
#   segment score = sum of (origin_population / n_destinations)
#                   for every origin whose optimal route passes through it.
# Segments are ranked highest to lowest -- those carrying the most people
# have the highest utilitarian priority for protection or intervention.
#
# Uses ra2ce.analysis.losses.traffic_analysis.TrafficAnalysis directly on
# the OD route GDF (opt_path column) and the origin_destination_table feather.
import pandas as pd
from ra2ce.analysis.losses.traffic_analysis.traffic_analysis import TrafficAnalysis

# 1. Load RA2CE intermediates written during the network build
od_table = pd.read_feather(ROOT_DIR / "static" / "output_graph" / "origin_destination_table.feather")
base_net = gpd.read_feather(ROOT_DIR / "static" / "output_graph" / "base_network.feather")
print(f"OD table : {len(od_table):,} rows   cols={list(od_table.columns)}")
print(f"Base net : {len(base_net):,} edges  sample cols={list(base_net.columns)[:10]}")

if not {"origin", "opt_path"}.issubset(od_gdf.columns):
    raise ValueError(
        "od_gdf is missing 'origin' or 'opt_path'. "
        "Re-run the OD analysis cell (Section 3) then reload with cell-od-load."
    )

# TrafficAnalysis hardcodes "values" as the population column name;
# RA2CE saves it under ORIGINS_COUNT_FIELD instead -- rename before passing in.
total_pop = int(od_table[ORIGINS_COUNT_FIELD].sum())
od_table_ta = od_table.rename(columns={ORIGINS_COUNT_FIELD: "values"})

# 2. Compute utilitarian traffic per network edge
ta = TrafficAnalysis(road_network=od_gdf, od_table=od_table_ta)
traffic_df = ta.optimal_route_od_link()
# Result cols: u (str), v (str), traffic (utilitarian), traffic_egalitarian
print(f"\nEdges with traffic : {len(traffic_df):,}")
print(f"Traffic range      : {traffic_df['traffic'].min():.2f} -- {traffic_df['traffic'].max():.2f}")

# 3. Join scores to base_network geometry (u/v are int in feather, str in traffic_df)
base_net["_u"] = base_net["u"].astype(str)
base_net["_v"] = base_net["v"].astype(str)
net_scored = base_net.merge(
    traffic_df[["u", "v", "traffic"]],
    left_on=["_u", "_v"], right_on=["u", "v"], how="left",
).drop(columns=["_u", "_v", "u", "v"], errors="ignore")

scored   = net_scored[net_scored["traffic"].notna()].copy()
unscored = net_scored[net_scored["traffic"].isna()].copy()
print(f"Scored edges : {len(scored):,}  |  Unscored: {len(unscored):,}")

# 4. Map
vmax_u = float(np.percentile(scored["traffic"], 95))
norm_u = mcolors.Normalize(vmin=0, vmax=vmax_u)
cmap_u = cm.inferno

BG = "#1a1a2e"
fig, ax = plt.subplots(figsize=(20, 20), facecolor=BG)
ax.set_facecolor(BG)

# Unscored segments as dim baseline
unscored.plot(ax=ax, color="#2a2a4e", linewidth=0.4, alpha=0.6)
# Scored segments: color = utilitarian traffic
scored.plot(ax=ax, column="traffic", cmap=cmap_u, norm=norm_u,
            linewidth=0.9, alpha=0.85, missing_kwds={"color": "#333"})
# Top-decile emphasis pass (thicker lines)
scored[scored["traffic"] >= np.percentile(scored["traffic"], 90)].plot(
    ax=ax, column="traffic", cmap=cmap_u, norm=norm_u, linewidth=3.0, alpha=1.0)

# Destination exit markers
try:
    dest_gdf.plot(ax=ax, color="#4cf0d8", marker="*", markersize=200, zorder=5)
    for _, row in dest_gdf.iterrows():
        ax.annotate(row.get("name", "Exit"), xy=(row.geometry.x, row.geometry.y),
                    xytext=(5, 3), textcoords="offset points",
                    color="#4cf0d8", fontsize=7.5, fontweight="bold")
except NameError:
    pass

ax.set_axis_off()
ax.set_title(
    "Tahoe Basin -- Utilitarian Network Criticality\n"
    "Segments ranked by population routed through them (POPULATION field)",
    color="#e8d5b7", fontsize=15, fontweight="bold", pad=15,
)

sm = cm.ScalarMappable(cmap=cmap_u, norm=norm_u)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Utilitarian traffic  (persons / destination)", color="#e8d5b7", fontsize=11)
cbar.ax.yaxis.set_tick_params(color="#e8d5b7")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="#e8d5b7")
cbar.outline.set_edgecolor("#e8d5b7")

rank_legend = [
    Line2D([0], [0], color="#2a2a4e", linewidth=1.5, alpha=0.8, label="No routed traffic"),
    Line2D([0], [0], color=cmap_u(0.25), linewidth=1.5, label="Lower utilitarian rank"),
    Line2D([0], [0], color=cmap_u(0.75), linewidth=1.5, label="Higher utilitarian rank"),
    Line2D([0], [0], color=cmap_u(1.0),  linewidth=3.0, label="Top 10% -- highest priority"),
]
ax.legend(handles=rank_legend, title="Utilitarian principle",
          facecolor="#2a2a4e", edgecolor="none",
          labelcolor="#e8d5b7", title_fontsize=10, fontsize=9, loc="lower right")

top10_thresh = np.percentile(scored["traffic"], 90)
ax.text(
    0.02, 0.02,
    f"Utilitarian principle: rank by persons using each link\n"
    f"Total origins population: {total_pop:,}\n"
    f"Top-10% threshold: {top10_thresh:.1f} persons / destination",
    transform=ax.transAxes, color="#e8d5b7", fontsize=10,
    verticalalignment="bottom",
    bbox=dict(facecolor="#2a2a4e", edgecolor="none", alpha=0.7, pad=5),
)

plt.tight_layout()
out_path = OUTPUT_PATH / "tahoe_od_utilitarian_ranking.png"
fig.savefig(out_path, dpi=300, facecolor=fig.get_facecolor(), bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

# Export for ArcGIS Pro as Shapefile (avoids GPKG SQL-type issues)
def _clean_for_shp(gdf):
    out = gdf.copy()
    for col in out.columns:
        if col == "geometry":
            continue
        if out[col].dtype == object:
            out[col] = out[col].fillna("").astype(str)
        elif out[col].dtype.kind == "f":
            out[col] = out[col].astype("float64")
        elif out[col].dtype.kind in ("i", "u"):
            out[col] = out[col].astype("int64")
    return out
out_shp = OUTPUT_PATH / "tahoe_od_utilitarian_ranking.shp"
_clean_for_shp(net_scored).to_file(out_shp)
print(f"Shapefile: {out_shp.resolve()}")


### Equity (Prioritarian) Network Criticality

Extends the utilitarian ranking by weighting each origin's population by a per-origin equity score (e.g., SVI, EJ index). Set `EQUITY_WEIGHT_FIELD` in the Configuration cell to the column name in the Origins layer before running.

In [ ]:
# ============================================================
# Equity (Prioritarian) Ranking -- population × per-origin equity weight
# ============================================================
# Implements the prioritarian principle from RA2CE equity analysis using a
# per-origin equity weight that lives directly on the Origins feature class,
# bypassing the normal equity-regions CSV workflow.
#
# How it works:
#   - RA2CE EquityAnalysis needs od_table["region"] and equity_data[region, weight]
#   - Since each origin already has its own weight, we set region = o_id (one
#     "region" per origin) and build equity_data from the Origins attributes.
#   - prioritarian_traffic per segment = sum(population * equity_weight / n_dests)
#     for all origins whose optimal route passes through it.
#   - Segments with high scores carry the most equity-weighted people.
#
# Set EQUITY_WEIGHT_FIELD in the Configuration cell before running.
import pandas as pd
from ra2ce.analysis.losses.traffic_analysis.equity_analysis import EquityAnalysis

if not EQUITY_WEIGHT_FIELD:
    raise ValueError(
        "EQUITY_WEIGHT_FIELD is not set. "
        "Add the column name for your per-origin equity weight to the Configuration cell."
    )

if not {"origin", "opt_path"}.issubset(od_gdf.columns):
    raise ValueError("od_gdf missing 'origin'/'opt_path' -- re-run OD analysis and reload.")

# 1. Load Origins from GDB to get equity weight column ---------------------
print(f"Reading '{ORIGINS_GDB_LAYER}' from GDB for equity weights ...")
orig_eq = gpd.read_file(NETWORK_GDB, layer=ORIGINS_GDB_LAYER)
print(f"  {len(orig_eq):,} origins  |  columns: {[c for c in orig_eq.columns if c != 'geometry']}")

if EQUITY_WEIGHT_FIELD not in orig_eq.columns:
    raise ValueError(
        f"'{EQUITY_WEIGHT_FIELD}' not found in Origins. "
        f"Available columns: {[c for c in orig_eq.columns if c != 'geometry']}"
    )

weight_series = orig_eq[EQUITY_WEIGHT_FIELD].reset_index(drop=True)
print(f"  Equity weight '{EQUITY_WEIGHT_FIELD}': "
      f"min={weight_series.min():.3f}  max={weight_series.max():.3f}  "
      f"mean={weight_series.mean():.3f}")

# 2. Build od_table with "values", "region", and equity_data ---------------
od_table = pd.read_feather(
    ROOT_DIR / "static" / "output_graph" / "origin_destination_table.feather"
)
base_net = gpd.read_feather(
    ROOT_DIR / "static" / "output_graph" / "base_network.feather"
)

od_table_eq = od_table.rename(columns={ORIGINS_COUNT_FIELD: "values"}).copy()

# o_id = "O_0", "O_1", ... maps to Origins row index 0, 1, ...
od_table_eq["o_idx"] = od_table_eq["o_id"].str[2:].astype(int)
od_table_eq["equity_weight"] = od_table_eq["o_idx"].map(weight_series.to_dict()).fillna(1.0)

# region = o_id (one "region" per origin); EquityAnalysis maps region → weight
od_table_eq["region"] = od_table_eq["o_id"]
equity_data = od_table_eq[["o_id", "equity_weight"]].rename(
    columns={"o_id": "region", "equity_weight": "weight"}
)

print(f"\nod_table_eq cols: {list(od_table_eq.columns)}")
print(f"equity_data sample:\n{equity_data.head(3).to_string(index=False)}")

# 3. Run EquityAnalysis ----------------------------------------------------
ea = EquityAnalysis(road_network=od_gdf, od_table=od_table_eq, equity_data=equity_data)
traffic_eq_df = ea.optimal_route_od_link()
# Cols: u, v, traffic (utilitarian), traffic_egalitarian, traffic_prioritarian
print(f"\nEdges scored      : {len(traffic_eq_df):,}")
print(f"Prioritarian range: {traffic_eq_df['traffic_prioritarian'].min():.2f} -- "
      f"{traffic_eq_df['traffic_prioritarian'].max():.2f}")

# 4. Join to base_network geometry -----------------------------------------
base_net["_u"] = base_net["u"].astype(str)
base_net["_v"] = base_net["v"].astype(str)
net_eq = base_net.merge(
    traffic_eq_df[["u", "v", "traffic_prioritarian"]],
    left_on=["_u", "_v"], right_on=["u", "v"], how="left",
).drop(columns=["_u", "_v", "u", "v"], errors="ignore")

scored_eq   = net_eq[net_eq["traffic_prioritarian"].notna()].copy()
unscored_eq = net_eq[net_eq["traffic_prioritarian"].isna()].copy()
print(f"Scored edges      : {len(scored_eq):,}  |  Unscored: {len(unscored_eq):,}")

# 5. Map -------------------------------------------------------------------
vmax_eq = float(np.percentile(scored_eq["traffic_prioritarian"], 95))
norm_eq  = mcolors.Normalize(vmin=0, vmax=vmax_eq)
cmap_eq  = cm.magma

BG = "#1a1a2e"
fig, ax = plt.subplots(figsize=(20, 20), facecolor=BG)
ax.set_facecolor(BG)

unscored_eq.plot(ax=ax, color="#2a2a4e", linewidth=0.4, alpha=0.6)
scored_eq.plot(ax=ax, column="traffic_prioritarian", cmap=cmap_eq, norm=norm_eq,
               linewidth=0.9, alpha=0.85, missing_kwds={"color": "#333"})
scored_eq[scored_eq["traffic_prioritarian"] >= np.percentile(
    scored_eq["traffic_prioritarian"], 90)].plot(
    ax=ax, column="traffic_prioritarian", cmap=cmap_eq, norm=norm_eq,
    linewidth=3.0, alpha=1.0)

try:
    dest_gdf.plot(ax=ax, color="#4cf0d8", marker="*", markersize=200, zorder=5)
    for _, row in dest_gdf.iterrows():
        ax.annotate(row.get("name", "Exit"), xy=(row.geometry.x, row.geometry.y),
                    xytext=(5, 3), textcoords="offset points",
                    color="#4cf0d8", fontsize=7.5, fontweight="bold")
except NameError:
    pass

ax.set_axis_off()
ax.set_title(
    f"Tahoe Basin -- Equity (Prioritarian) Network Criticality\n"
    f"Segments ranked by population × {EQUITY_WEIGHT_FIELD}",
    color="#e8d5b7", fontsize=15, fontweight="bold", pad=15,
)

sm = cm.ScalarMappable(cmap=cmap_eq, norm=norm_eq)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label(f"Prioritarian traffic  (pop × {EQUITY_WEIGHT_FIELD} / destination)",
               color="#e8d5b7", fontsize=10)
cbar.ax.yaxis.set_tick_params(color="#e8d5b7")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="#e8d5b7")
cbar.outline.set_edgecolor("#e8d5b7")

rank_legend = [
    Line2D([0], [0], color="#2a2a4e", linewidth=1.5, alpha=0.8, label="No routed traffic"),
    Line2D([0], [0], color=cmap_eq(0.25), linewidth=1.5, label="Lower equity-weighted rank"),
    Line2D([0], [0], color=cmap_eq(0.75), linewidth=1.5, label="Higher equity-weighted rank"),
    Line2D([0], [0], color=cmap_eq(1.0),  linewidth=3.0, label="Top 10% -- highest priority"),
]
ax.legend(handles=rank_legend, title="Prioritarian principle",
          facecolor="#2a2a4e", edgecolor="none",
          labelcolor="#e8d5b7", title_fontsize=10, fontsize=9, loc="lower right")

total_pop_eq = int(od_table_eq["values"].sum())
top10_eq = np.percentile(scored_eq["traffic_prioritarian"], 90)
ax.text(
    0.02, 0.02,
    f"Prioritarian principle: rank by population × equity weight\n"
    f"Equity weight field: {EQUITY_WEIGHT_FIELD}\n"
    f"Total origins population: {total_pop_eq:,}\n"
    f"Top-10% threshold: {top10_eq:.2f}",
    transform=ax.transAxes, color="#e8d5b7", fontsize=10,
    verticalalignment="bottom",
    bbox=dict(facecolor="#2a2a4e", edgecolor="none", alpha=0.7, pad=5),
)

plt.tight_layout()
out_path = OUTPUT_PATH / "tahoe_od_equity_prioritarian.png"
fig.savefig(out_path, dpi=300, facecolor=fig.get_facecolor(), bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

# Export for ArcGIS Pro as Shapefile (avoids GPKG SQL-type issues)
def _clean_for_shp(gdf):
    out = gdf.copy()
    for col in out.columns:
        if col == "geometry":
            continue
        if out[col].dtype == object:
            out[col] = out[col].fillna("").astype(str)
        elif out[col].dtype.kind == "f":
            out[col] = out[col].astype("float64")
        elif out[col].dtype.kind in ("i", "u"):
            out[col] = out[col].astype("int64")
    return out
out_shp = OUTPUT_PATH / "tahoe_od_equity_prioritarian.shp"
_clean_for_shp(net_eq).to_file(out_shp)
print(f"Shapefile: {out_shp.resolve()}")


## 4. OD Accessibility with Hazard Disruption

Overlays a hazard raster on the network, then runs multi-link OD analysis to find the best accessible route for each origin when hazard-exposed links are disrupted.

**Required inputs** (in addition to Origins/Destinations from Section 3):
- A hazard raster (GeoTIFF) placed in `static/network/`
- `HAZARD_RASTER` and `HAZARD_CRS` set in the Configuration cell

RA2CE reprojects the graph into the raster CRS for the intersection, then converts results back to EPSG:4326. Set `HAZARD_CRS` to match the native projection of your raster.

In [ ]:
network_section_hz = NetworkSection(
    source=SourceEnum.SHAPEFILE,
    primary_file=NETWORK_SHP,
    save_gpkg=True,
    reuse_network_output=True,
)

od_section_hz = OriginsDestinationsSection(
    origins=NETWORK_PATH / ORIGINS_SHP,
    destinations=NETWORK_PATH / DESTINATIONS_SHP,
    origin_count=ORIGINS_COUNT_FIELD,
)

hazard_section = HazardSection(
    hazard_map=[NETWORK_PATH / HAZARD_RASTER],
    aggregate_wl=AggregateWlEnum.MEAN,
    hazard_crs=HAZARD_CRS,
)

network_config_hz = NetworkConfigData(
    root_path=ROOT_DIR,
    static_path=ROOT_DIR / "static",
    crs=CRS.from_epsg(4326),
    network=network_section_hz,
    origins_destinations=od_section_hz,
    hazard=hazard_section,
)

od_hazard_analysis = AnalysisSectionLosses(
    name="tahoe_od_hazard",
    analysis=AnalysisLossesEnum.MULTI_LINK_ORIGIN_CLOSEST_DESTINATION,
    weighing=WeighingEnum.LENGTH,
    calculate_route_without_disruption=True,
    save_csv=True,
    save_gpkg=True,
)

od_hazard_config = AnalysisConfigData(
    output_path=OUTPUT_PATH,
    static_path=ROOT_DIR / "static",
    analyses=[od_hazard_analysis],
)

handler_hz = Ra2ceHandler.from_config(network=network_config_hz, analysis=od_hazard_config)
handler_hz.configure()
handler_hz.run_analysis()

In [ ]:
hz_output_path = OUTPUT_PATH / "multi_link_origin_closest_destination"
origins_gpkg   = hz_output_path / "tahoe_od_hazard_origins.gpkg"

origin_gdf = gpd.read_file(origins_gpkg)
print(f"Origins: {len(origin_gdf):,}  |  Columns: {list(origin_gdf.columns)}")
origin_gdf.head()

In [ ]:
# Interactive map: origin accessibility colored by hazard-adjusted extra distance.
# Column names vary by RA2CE version -- inspect origin_gdf.columns if EV1_me_A is absent.
map_col = next(
    (c for c in origin_gdf.columns if c.startswith("EV") and "_me_" in c),
    None,
)
if map_col:
    m = origin_gdf.explore(
        column=map_col,
        cmap=["green", "red"],
        marker_kwds={"radius": 5},
        tiles="CartoDB dark_matter",
    )
    display(m)
else:
    print("No EV accessibility column found. Available columns:", list(origin_gdf.columns))